# 📅 11 · 时间序列与指标监控

> 业务指标（DAU/收入/转化率）是**时间序列**：有趋势、有季节、有噪声。本节讲**趋势/季节分解、移动平均、同比环比、3σ 异常检测**，服务于「指标监控 + 异常预警」这一数据分析师日常。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 数据里有趋势/季节/噪声吗？ | 分解 |
| 2 | 怎么平滑掉噪声看趋势？ | 移动平均 |
| 3 | 这个月比上个月/去年怎么样？ | 环比/同比 |
| 4 | 今天的值异常吗？ | 3σ / z-score 预警 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ 构造含趋势+季节+噪声的日活数据

In [2]:
rng = np.random.default_rng(31)
n_days = 90
dates = pd.date_range('2025-01-01', periods=n_days, freq='D')
trend = np.linspace(10000, 13000, n_days)                    # 缓慢上升趋势
season = 800 * np.sin(2*np.pi*np.arange(n_days)/7)           # 周季节
noise = rng.normal(0, 300, n_days)                           # 随机噪声
dau = trend + season + noise
ts = pd.Series(dau, index=dates, name='DAU')
print(ts.head())
print('\n均值:', ts.mean().round(0), ' 标准差:', ts.std().round(0))

2025-01-01     9881.409613
2025-01-02    10738.347517
2025-01-03    11029.496541
2025-01-04    10156.582647
2025-01-05    10018.023745
Freq: D, Name: DAU, dtype: float64

均值: 11487.0  标准差: 1063.0


## 2️⃣ 移动平均平滑（7 日窗口）

In [3]:
ma7 = ts.rolling(7, center=True).mean()

plt.figure(figsize=(10, 4))
plt.plot(ts.index, ts.values, alpha=0.4, label='原始 DAU')
plt.plot(ma7.index, ma7.values, color='red', lw=2, label='7 日移动平均')
plt.legend(); plt.title('DAU 与 7 日移动平均')
plt.ylabel('用户数')
plt.show()
assert not ma7.isna().all()
print('✅ 移动平均抹平噪声后趋势更清晰')

✅ 移动平均抹平噪声后趋势更清晰


C:\Users\24260\AppData\Local\Temp\ipykernel_46624\2664760674.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3️⃣ 环比 / 同比（月度视角）

In [4]:
# 转月度并计算环比
monthly = ts.resample('ME').mean()
mom = monthly.pct_change() * 100
out = pd.DataFrame({'DAU均值': monthly.round(0), '环比%': mom.round(2)})
print(out)
assert (mom.dropna() > -100).all()
print('\n➡️ 环比 = (本月-上月)/上月；同比需上年同月数据')

              DAU均值   环比%
2025-01-31  10537.0   NaN
2025-02-28  11425.0  8.43
2025-03-31  12493.0  9.34

➡️ 环比 = (本月-上月)/上月；同比需上年同月数据


## 4️⃣ 3σ 异常检测（滚动窗口）

In [5]:
# 在第 50 天人为制造一个异常尖峰
ts2 = ts.copy()
ts2.iloc[49] += 2200

# 用「过去 14 天」估计（shift 避免尖峰污染自己的检测窗口）
roll_mean = ts2.rolling(14, min_periods=14).mean().shift(1)
roll_std  = ts2.rolling(14, min_periods=14).std().shift(1)
upper = roll_mean + 3*roll_std
alarm = (ts2 > upper)

plt.figure(figsize=(10, 4))
plt.plot(ts2.index, ts2.values, label='DAU（含第50天尖峰）')
plt.plot(upper.index, upper.values, 'r--', label='+3σ 上界（上一天窗口）')
plt.scatter(ts2.index[alarm], ts2.values[alarm], color='red', s=60, label='触发预警')
plt.legend(); plt.title('3σ 滚动异常检测（窗口前移，防自身污染）')
plt.show()
print('触发预警的天数:', int(alarm.sum()))
assert bool(alarm.iloc[49]) is True  # 第 50 天（index 49）触发
print('✅ 正确捕获人为尖峰，无自身污染')

触发预警的天数: 1
✅ 正确捕获人为尖峰，无自身污染


C:\Users\24260\AppData\Local\Temp\ipykernel_46624\2574042291.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5️⃣ 指标监控看板维度（必背）

- **趋势**：日/周/月粒度切换，看长期走向；
- **异常**：3σ、IQR、环比阈值（如 -15%）、同比阈值，多规则组合；
- **分层**：按渠道/版本/城市下钻，定位异常来源；
- **时效**：实时/小时/日报分级监控；
- **阈值设定**：结合历史分位数（如 P99）与业务容忍度；
- **预警动作**：自动通知 + 根因排查清单 + 结论回填。

## 📝 自测清单

- [ ] 能手写移动平均并说明窗口选择
- [ ] 能手写环比/同比计算
- [ ] 能手写 3σ 滚动异常检测并解释参数
- [ ] 能说出监控看板的五个维度
- [ ] 能解释趋势/季节/噪声三要素